# ISIC 2019 Bronze ingestion sample

This notebook exercises the Bronze layer with a small synthetic sample.

Goals:
- verify the Bronze storage layout exists
- write a small ingestion run record
- write a small source-metadata sample into `bronze.isic_2019_source_metadata`
- confirm the expected paths are reachable from Databricks

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4

from isic_pipeline.bootstrap import build_layout, ensure_bronze_tables, load_dataset_config

DATASET_CONFIG = load_dataset_config(Path("config/datasets/isic_2019.yaml"))
layout = build_layout(DATASET_CONFIG)
bronze_paths = layout["bronze_paths"]
bronze_tables = layout["bronze_tables"]

print(bronze_paths)
print(bronze_tables)


In [ ]:
ensure_bronze_tables(spark)
for path in bronze_paths.values():
    dbutils.fs.mkdirs(path)

dbutils.fs.mkdirs(f"{bronze_paths['metadata']}/sample")
dbutils.fs.mkdirs(f"{bronze_paths['ingestion_runs']}/sample")

display(spark.sql("SHOW TABLES IN bronze"))


In [ ]:
ingestion_run_id = f"bronze-sample-{uuid4().hex[:12]}"
started_at = datetime.now(timezone.utc)
finished_at = datetime.now(timezone.utc)

sample_metadata = [
    {
        "image_id": "ISIC_0000000",
        "source_uri": f"{bronze_paths['images']}/ISIC_0000000.jpg",
        "patient_id": "patient-001",
        "lesion_id": "lesion-001",
        "label_raw": "melanoma",
        "label_source": "sample_fixture",
        "label_normalized": "melanoma",
        "acquisition_date": "2019-01-01",
        "source_checksum": "sample-checksum-0001",
        "ingestion_run_id": ingestion_run_id,
        "ingested_at": started_at,
    },
    {
        "image_id": "ISIC_0000001",
        "source_uri": f"{bronze_paths['images']}/ISIC_0000001.jpg",
        "patient_id": "patient-002",
        "lesion_id": "lesion-002",
        "label_raw": "nevus",
        "label_source": "sample_fixture",
        "label_normalized": "nevus",
        "acquisition_date": "2019-01-02",
        "source_checksum": "sample-checksum-0002",
        "ingestion_run_id": ingestion_run_id,
        "ingested_at": started_at,
    },
]

spark.createDataFrame(sample_metadata).write.mode("append").saveAsTable("bronze.isic_2019_source_metadata")

run_row = [{
    "ingestion_run_id": ingestion_run_id,
    "dataset_name": DATASET_CONFIG["dataset_name"],
    "source_version": DATASET_CONFIG["source_version"],
    "started_at": started_at,
    "finished_at": finished_at,
    "status": "success",
    "records_seen": 2,
    "records_written": 2,
}]

spark.createDataFrame(run_row).write.mode("append").saveAsTable("bronze.ingestion_runs")

display(spark.table("bronze.isic_2019_source_metadata"))


In [ ]:
print("Metadata path contents:")
display(dbutils.fs.ls(bronze_paths["metadata"]))

print("Images path contents:")
display(dbutils.fs.ls(bronze_paths["images"]))

print("Ingestion runs path contents:")
display(dbutils.fs.ls(bronze_paths["ingestion_runs"]))

print("Recent ingestion runs:")
display(spark.table("bronze.ingestion_runs").orderBy("started_at", ascending=False))


## Next step

Replace the synthetic rows with a real small sample from the ISIC 2019 source metadata and a few downloaded images once storage verification is working.